In [0]:
# Carregamento das credenciais

import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv()

if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"Variáveis encontradas: {dotenv_path}")
else:
    print("Variáveis não encontradas.")

credenciais = {
    "SQL_HOST": os.getenv("SQL_HOST"),
    "SQL_DATABASE": os.getenv("SQL_DATABASE"),
    "SQL_USERNAME": os.getenv("SQL_USERNAME"),
    "SQL_PASSWORD": os.getenv("SQL_PASSWORD"),
}

print("\nStatus das Credenciais SQL Server")
for k, v in credenciais.items():
    print(f"  {k}: {'[DEFINIDO]' if v else '[NÃO DEFINIDO]'}")

if not all(credenciais.values()):
    raise ValueError("Existem credenciais do SQL Server pendentes no .env!")

SQL_HOST = credenciais["SQL_HOST"]
SQL_DATABASE = credenciais["SQL_DATABASE"]
SQL_USERNAME = credenciais["SQL_USERNAME"]
SQL_PASSWORD = credenciais["SQL_PASSWORD"]
SQL_PORT = "1433"


# Teste de leitura (JDBC para leitura no serverless)

jdbc_url = (
    f"jdbc:sqlserver://{SQL_HOST}:{SQL_PORT};"
    f"database={SQL_DATABASE};"
    "encrypt=true;trustServerCertificate=false;loginTimeout=30"
)

connection_properties = {
    "user": SQL_USERNAME,
    "password": SQL_PASSWORD,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

try:
    df_teste_leitura = spark.read.jdbc(
        url=jdbc_url,
        table="(SELECT 1 AS teste) AS t",
        properties=connection_properties,
    )
    df_teste_leitura.show()
    print("Conexão JDBC (leitura) funcionando.")
except Exception as e:
    print("Falha na leitura JDBC:")
    print(e)


# Teste de escrita (CREATE/INSERT) 

dados_teste = pd.DataFrame({
    "id": [1, 2, 3],
    "mensagem": ["teste", "permissao", "escrita"],
})

df_spark_teste = spark.createDataFrame(dados_teste)

try:
    (
        df_spark_teste.write
        .format("sqlserver")
        .mode("overwrite")
        .option("host", SQL_HOST)
        .option("port", SQL_PORT)
        .option("database", SQL_DATABASE)
        .option("dbtable", "dbo.teste_permissao_escrita")
        .option("user", SQL_USERNAME)
        .option("password", SQL_PASSWORD)
        .save()
    )
    print("Permissão de CREATE/INSERT confirmada — tabela criada com sucesso.")
except Exception as e:
    print("Falha ao tentar escrever no banco:")
    print(e)

Variáveis encontradas: /Workspace/Users/eng.stephanni@gmail.com/.env

Status das Credenciais SQL Server
  SQL_HOST: [DEFINIDO]
  SQL_DATABASE: [DEFINIDO]
  SQL_USERNAME: [DEFINIDO]
  SQL_PASSWORD: [DEFINIDO]
+-----+
|teste|
+-----+
|    1|
+-----+

Conexão JDBC (leitura) funcionando.
Permissão de CREATE/INSERT confirmada — tabela criada com sucesso.
